## Tokenization by using Bag of Word

In [1]:
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import json
import pandas as pd
import spellchecker

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
# Open the cleaned mendeley data, tokenize by nltk, check spelling, lemmatize 
with open("../cleaned_data/mendeley_cleaned.json", "r") as f:
    mendeley_data = json.load(f)

BOW = {}
# spell_checker = spellchecker.SpellChecker()
# potentially_misspelled = set()

for review in mendeley_data:
    tokens = word_tokenize(str(review["review"]).lower())
    for token in tokens:
        # check for . in the middle; if yes, split then the rest add to the list
        if "." in token:
            splitted_tok = token.split(".")
            token = splitted_tok[0]
            for i in range(1, len(splitted_tok)):
                tokens.append(splitted_tok[i])
        # check for alphanumeric only
        if not token.isalnum():
            continue
        if token not in BOW:
             # add for potentially misspelled words for later checking
            # if token not in spell_checker:
            #     potentially_misspelled.add(token)
            BOW[token] = 1
        else:
            BOW[token] += 1

# THIS IS TAKING TOO LONG
# # See if corrected version exists to avoid false positives
# to_check = [tok for tok in potentially_misspelled if BOW[tok] < 5]
# print(f"Total potentially misspelled tokens: {len(to_check)}")
# for token in to_check:
#     corrected = spell_checker.correction(token)
#     if corrected in BOW:
#         BOW[corrected] += BOW[token]
#         del BOW[token]


BOW_to_json = [{"token": token, "count": count} for token, count in BOW.items() if token not in stopwords.words('english')]
BOW_to_json = sorted(BOW_to_json, key=lambda x: x["count"], reverse=True)

print(f"Total unique tokens: {len(BOW_to_json)}")
print("Top 10 tokens:")
for i in range(10):
    print(f"\t{BOW_to_json[i]['token']}: {BOW_to_json[i]['count']}")

# dump BOW to a json file
with open("tokens/BOW.json", "w") as f:
    json.dump(BOW_to_json, f, indent=4)

Total unique tokens: 54735
Top 10 tokens:
	game: 88792
	like: 18239
	play: 15795
	get: 14316
	good: 13321
	fun: 11874
	time: 11755
	one: 10568
	even: 9992
	really: 9982


In [7]:
# Check how many bad words are in the BOW
with open("../cleaned_data/bad_words.txt", "r") as f:
    bad_words = f.read().splitlines()

occuring_bad_words = [token for token in BOW if token in bad_words]
print(f"Total bad words in BOW: {len(occuring_bad_words)}")
print(f"Percentage of bad words in BOW: {len(occuring_bad_words) / len(BOW) * 100:.2f}%")

Total bad words in BOW: 778
Percentage of bad words in BOW: 1.42%
